# Ch 08. 활성화 함수와 손실 함수 — 해설

> 이 노트북은 다섯 연습문제의 엄밀한 해설과 재현 가능한 검증을 포함합니다.

## 문제 1

**문제**: Sigmoid의 도함수가 $\sigma(x)(1-\sigma(x))$임을 직접 유도하라.

### 엄밀한 해설

통제 변수: **derivative formula versus central difference**.  측정 지표: **maximum absolute derivative error**.  해석과 한계: Differentiating 1/(1+e^-x) gives sigma(x)(1-sigma(x)); numerical differences validate the identity away from floating-point saturation.


In [ ]:
import torch
x=torch.linspace(-6,6,101,dtype=torch.float64); s=torch.sigmoid(x); e=1e-5; num=(torch.sigmoid(x+e)-torch.sigmoid(x-e))/(2*e); err=(num-s*(1-s)).abs().max().item(); print({'max_derivative_error':err}); assert err<1e-9


## 문제 2

**문제**: ReLU의 "죽은 ReLU" 문제를 실험으로 보이라 (큰 음수 가중치 → 뉴런이 영원히 0).

### 엄밀한 해설

통제 변수: **bias: strongly negative versus positive**.  측정 지표: **fraction of zero activations and gradient norm**.  해석과 한계: With all preactivations negative, ReLU outputs and derivatives are zero, so gradient descent cannot revive that neuron on these data.


In [ ]:
import torch
x=torch.ones(32,4); w=torch.full((4,1),-5.,requires_grad=True); b=torch.tensor(-5.,requires_grad=True); a=torch.relu(x@w+b); a.sum().backward(); print({'zero_fraction':(a==0).float().mean().item(),'weight_grad_norm':w.grad.norm().item()}); assert w.grad.norm()==0


## 문제 3

**문제**: GELU와 ReLU의 차이가 깊은 MLP 학습에 미치는 영향을 실험하라.

### 엄밀한 해설

통제 변수: **activation: ReLU versus GELU**.  측정 지표: **final loss and first-layer gradient norm in a matched deep MLP**.  해석과 한계: Weights, data, optimizer, and steps are matched. GELU preserves small negative signals while ReLU truncates them; results are specific to this reduced task.


In [ ]:
import torch
torch.manual_seed(80); X=torch.randn(128,12); y=(X[:,:4].sum(1)>0).long(); seed=torch.nn.Sequential(torch.nn.Linear(12,24),torch.nn.Linear(24,24),torch.nn.Linear(24,2)); state=seed.state_dict()
for name,act in [('relu',torch.relu),('gelu',torch.nn.functional.gelu)]:
 m=torch.nn.Sequential(torch.nn.Linear(12,24),torch.nn.Linear(24,24),torch.nn.Linear(24,2)); m.load_state_dict(state); o=torch.optim.Adam(m.parameters(),lr=.02)
 for _ in range(80): h=act(m[0](X)); h=act(m[1](h)); loss=torch.nn.functional.cross_entropy(m[2](h),y); o.zero_grad(); loss.backward(); o.step()
 print({'activation':name,'final_loss':loss.item(),'first_grad_norm':m[0].weight.grad.norm().item()})


## 문제 4

**문제**: MSE와 CE의 그래디언트를 비교하고, 분류 문제에 CE가 더 적합한 이유를 설명하라.

### 엄밀한 해설

통제 변수: **loss: softmax cross-entropy versus MSE on probabilities**.  측정 지표: **correct-class logit gradient magnitude**.  해석과 한계: CE has gradient p-y and avoids the extra sigmoid/softmax Jacobian factor that attenuates MSE gradients when confidently wrong.


In [ ]:
import torch
z=torch.tensor([[-5.,5.]],requires_grad=True); y=torch.tensor([0]); ce=torch.nn.functional.cross_entropy(z,y); ce.backward(); gce=z.grad.clone(); z.grad=None; p=z.softmax(1); mse=((p-torch.tensor([[1.,0.]]))**2).mean(); mse.backward(); gmse=z.grad.clone(); print({'CE_grad':gce.tolist(),'MSE_grad':gmse.tolist(),'ratio':(gce.abs().max()/gmse.abs().max()).item()}); assert gce.abs().max()>gmse.abs().max()


## 문제 5

**문제**: Label smoothing $\epsilon = 0, 0.1, 0.3$에 대해 MNIST MLP 학습 정확도를 비교하라.

### 엄밀한 해설

통제 변수: **label-smoothing epsilon: 0, 0.1, 0.3**.  측정 지표: **test accuracy and mean confidence**.  해석과 한계: All conditions share a fixed synthetic classification split and initialization. Smoothing changes targets and calibration pressure; the reduced result is not presented as MNIST performance.


In [ ]:
import torch
torch.manual_seed(81); X=torch.randn(360,10); y=(X[:,:3].sum(1)>0).long(); tr=slice(0,300); te=slice(300,None); base=torch.nn.Linear(10,2).state_dict()
for eps in (0.,.1,.3):
 m=torch.nn.Linear(10,2); m.load_state_dict(base); o=torch.optim.SGD(m.parameters(),lr=.2)
 for _ in range(80): loss=torch.nn.functional.cross_entropy(m(X[tr]),y[tr],label_smoothing=eps); o.zero_grad(); loss.backward(); o.step()
 p=m(X[te]).softmax(1); print({'epsilon':eps,'accuracy':(p.argmax(1)==y[te]).float().mean().item(),'mean_confidence':p.max(1).values.mean().item()})
